## Шаг 0. Клонирование репозитория (Colab / Kaggle)

На Colab и Kaggle локальной копии репо нет — первой ячейкой клонируем его и добавляем `src/` в `sys.path`, чтобы `from gp1.env import ...` работал. Локально ячейка — no-op.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/DKazhekin/ITMO_Speech_Recognition_Course.git"
REPO_NAME = "ITMO_Speech_Recognition_Course"

_in_colab = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ
_in_kaggle = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if _in_colab or _in_kaggle:
    repo_dir = Path(REPO_NAME)
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    gp1_src = (repo_dir / "group-projects/gp1/src").resolve()
    if str(gp1_src) not in sys.path:
        sys.path.insert(0, str(gp1_src))
    print(f"Repo cloned, gp1 src on sys.path: {gp1_src}")
else:
    print("Local run — repo already checked out.")

# 03. EfficientConformer — обучение char-vocab ASR

Self-contained ноутбук: от сырых данных до submission.
Архитектура: EfficientConformer (3-стадийный ConformerEncoder с inter-stage downsampling, ~4.2M параметров).
HP Random Search оборачивает весь цикл обучения.

In [ ]:
from gp1.env import setup_environment

paths, device = setup_environment()
print("device:", device)
print("train_csv:", paths.train_csv)
print("dev_csv:", paths.dev_csv)
print("ckpt_root:", paths.ckpt_root)


## Шаг 1. Импорты

In [ ]:
import json
import random
import time

import torch
from torch.utils.data import DataLoader

from gp1.data.dataset import SpokenNumbersDataset
from gp1.data.collate import collate_fn
from gp1.data.audio_aug import AudioAugmenter
from gp1.data.spec_aug import SpecAugmenter
from gp1.data.manifest import records_from_csv
from gp1.features.melbanks import LogMelFilterBanks
from gp1.losses.ctc import CTCLoss
from gp1.models.efficient_conformer import EfficientConformer
from gp1.text.vocab import CharVocab
from gp1.text.denormalize import words_to_digits
from gp1.train.trainer import Trainer, TrainerConfig
from gp1.train.optim import build_adamw
from gp1.train.schedulers import build_noam
from gp1.train.checkpoint import load_checkpoint
from gp1.train.metrics import compute_cer, compute_per_speaker_cer
from gp1.decoding.greedy import greedy_decode
from gp1.types import AugConfig

## Шаг 2. Гиперпараметры (FIXED + HP_GRID)

In [ ]:
FIXED = {
    "samplerate": 16000,
    "n_fft": 512,
    "n_mels": 80,
    "hop_length": 160,
    "win_length": 400,
    "max_epochs": 80,
    "grad_accum": 2,
    "subsample_factor": 4,
    "n_heads": 4,
    "ff_ratio": 4,
}
HP_GRID = {
    "lr":                        [5e-4, 3e-4, 1e-4],
    "weight_decay":              [1e-6, 1e-4],
    "dropout":                   [0.05, 0.1, 0.15],
    "conv_kernel":               [15, 31],
    "d_model_stages":            [[96, 128, 128], [96, 144, 144]],
    "n_blocks_per_stage":        [[4, 4, 4], [3, 3, 3]],
    "warmup_steps":              [5000, 10000, 15000],
    "specaug_freq_mask_param":   [15, 20],
    "specaug_time_mask_param":   [25, 35],
    "speed_prob":                [0.5, 1.0],
    "noise_prob":                [0.0, 0.3],
}
N_TRIALS = 6
SEED = 42
print("FIXED:", FIXED)
print("N_TRIALS:", N_TRIALS)


## Шаг 3. Загрузка записей из CSV

In [ ]:
train_records = records_from_csv(paths.train_csv, paths.train_root)
dev_records = records_from_csv(paths.dev_csv, paths.dev_root)
print(f"Train records: {len(train_records)}, Dev records: {len(dev_records)}")


## Шаг 4. Vocabulary

In [ ]:
vocab = CharVocab()
print(f"Vocab size: {vocab.vocab_size}, blank_id: {vocab.blank_id}")


## Шаг 5. HP Random Search — Training Loop

In [ ]:
SEED = 42
random.seed(SEED)
trial_results = []
run_root = paths.ckpt_root / f"03_efficient_conformer_{int(time.time())}"
run_root.mkdir(parents=True, exist_ok=True)

for trial in range(N_TRIALS):
    hp = {k: random.choice(v) for k, v in HP_GRID.items()}
    print(f"\n=== Trial {trial + 1}/{N_TRIALS} ===")
    print(json.dumps({**FIXED, **hp}, default=str, indent=2))

    aug_cfg = AugConfig(
        speed_prob=hp["speed_prob"],
        noise_prob=hp["noise_prob"],
        seed=SEED + trial,
    )
    train_aug = AudioAugmenter(aug_cfg)
    spec_aug = SpecAugmenter(
        freq_mask_param=hp["specaug_freq_mask_param"],
        num_freq_masks=2,
        time_mask_param=hp["specaug_time_mask_param"],
        num_time_masks=5,
        time_mask_max_ratio=0.05,
        seed=SEED + trial,
    )

    train_ds = SpokenNumbersDataset(
        records=train_records,
        vocab=vocab,
        augmenter=train_aug,
        target_samplerate=FIXED["samplerate"],
    )
    dev_ds = SpokenNumbersDataset(
        records=dev_records,
        vocab=vocab,
        augmenter=None,
        target_samplerate=FIXED["samplerate"],
    )
    train_loader = DataLoader(
        train_ds,
        batch_size=16,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=2,
    )
    dev_loader = DataLoader(
        dev_ds,
        batch_size=16,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=2,
    )

    d_stages = hp["d_model_stages"]
    n_stages = hp["n_blocks_per_stage"]
    model = EfficientConformer(
        vocab_size=vocab.vocab_size,
        d_model_stages=tuple(d_stages),
        n_blocks_per_stage=tuple(n_stages),
        n_heads=FIXED["n_heads"],
        ff_ratio=FIXED["ff_ratio"],
        conv_kernel=hp["conv_kernel"],
        dropout=hp["dropout"],
    ).to(device)

    ctc = CTCLoss(blank_id=vocab.blank_id)
    optimizer = build_adamw(
        model.parameters(),
        lr=hp["lr"],
        weight_decay=hp["weight_decay"],
    )
    # Use mean d_model for Noam scaling.
    d_model_mean = int(sum(d_stages) / len(d_stages))
    scheduler = build_noam(
        optimizer,
        d_model=d_model_mean,
        warmup_steps=hp["warmup_steps"],
    )

    trial_dir = run_root / f"trial_{trial:02d}"
    cfg = TrainerConfig(
        max_epochs=FIXED["max_epochs"],
        grad_accum=FIXED["grad_accum"],
        fp16_autocast=True,
        val_every_n_epochs=1,
        early_stop_patience=10,
        early_stop_metric="max_speaker_cer",
        ckpt_dir=trial_dir,
    )
    trainer = Trainer(
        model=model,
        ctc_loss=ctc,
        optimizer=optimizer,
        scheduler=scheduler,
        train_loader=train_loader,
        val_loader=dev_loader,
        vocab=vocab,
        config=cfg,
        device=device,
        audio_cfg={
            "n_fft": FIXED["n_fft"],
            "n_mels": FIXED["n_mels"],
            "hop_length": FIXED["hop_length"],
            "win_length": FIXED["win_length"],
            "samplerate": FIXED["samplerate"],
        },
        spec_augmenter=spec_aug,
    )
    result = trainer.fit()
    best_ckpt = result["best_ckpt_path"]
    trial_results.append({"trial": trial, "hp": hp, **result})

    if best_ckpt is not None:
        with open(trial_dir / "meta.json", "w") as f:
            json.dump(
                {
                    "arch": "efficient_conformer",
                    "hparams": {**FIXED, **hp},
                    "val_cer": result["best_val_cer"],
                    "trial": trial,
                },
                f,
                indent=2,
                default=str,
            )

print("\nHP search complete.")


## Шаг 6. Сводный отчёт трайлов

In [ ]:
trial_results_sorted = sorted(trial_results, key=lambda r: r["best_val_cer"])
print(f"{'trial':>5} {'best_val_cer':>12} {'lr':>8} {'dropout':>8} {'conv_kernel':>12}")
print("-" * 55)
for r in trial_results_sorted:
    hp = r["hp"]
    print(
        f"{r['trial']:>5}"
        f" {r['best_val_cer']:>12.4f}"
        f" {hp['lr']:>8.5f}"
        f" {hp['dropout']:>8.4f}"
        f" {hp['conv_kernel']:>12}"
    )

## Шаг 7. Валидация лучшей модели на dev + greedy predictions

In [ ]:
best_result = trial_results_sorted[0]
best_ckpt = best_result["best_ckpt_path"]
best_hp = best_result["hp"]
print("Best trial val_cer:", best_result["best_val_cer"], "ckpt:", best_ckpt)

d_stages = best_hp["d_model_stages"]
n_stages = best_hp["n_blocks_per_stage"]
model = EfficientConformer(
    vocab_size=vocab.vocab_size,
    d_model_stages=tuple(d_stages),
    n_blocks_per_stage=tuple(n_stages),
    n_heads=FIXED["n_heads"],
    ff_ratio=FIXED["ff_ratio"],
    conv_kernel=best_hp["conv_kernel"],
    dropout=best_hp["dropout"],
).to(device)
meta = load_checkpoint(best_ckpt, model)
model.eval()

mel_extractor = LogMelFilterBanks(
    n_fft=FIXED["n_fft"],
    samplerate=FIXED["samplerate"],
    hop_length=FIXED["hop_length"],
    win_length=FIXED["win_length"],
    n_mels=FIXED["n_mels"],
).to(device)

dev_ds_eval = SpokenNumbersDataset(
    records=dev_records, vocab=vocab, augmenter=None, target_samplerate=FIXED["samplerate"]
)
dev_loader_eval = DataLoader(
    dev_ds_eval, batch_size=32, shuffle=False, collate_fn=collate_fn, num_workers=2
)

predictions, refs, spks = [], [], []
with torch.no_grad():
    for batch in dev_loader_eval:
        audio = batch.audio.to(device)
        audio_lengths = batch.audio_lengths.to(device)
        mel = mel_extractor(audio)
        mel_lengths = (audio_lengths // FIXED["hop_length"] + 1).clamp(max=mel.size(-1)).long()
        out = model(mel, mel_lengths)
        hyps = greedy_decode(out.log_probs, out.output_lengths, vocab)
        predictions.extend(hyps)
        refs.extend(batch.transcriptions)
        spks.extend(batch.spk_ids)

digit_preds = [words_to_digits(h) for h in predictions]
cer = compute_cer(refs, digit_preds)
per_spk = compute_per_speaker_cer(refs, digit_preds, spks)
print(f"Dev CER (greedy): {cer:.4f}")
print("Per-speaker CER:", per_spk)

## Шаг 8 (опционально). Beam search + KenLM rescore

In [ ]:
# Optional — requires kenlm + pyctcdecode and a trained LM model.
# See notebooks/experiments/06_kenlm_beam_rescore.ipynb for the full pipeline.
# from gp1.decoding.beam_pyctc import BeamSearchDecoder, BeamSearchConfig


## Шаг 9. Submission (если test данные доступны)

In [ ]:
if paths.test_root is not None:
    from gp1.submit.inference_utils import build_test_dataloader, write_submission

    test_records = records_from_csv(paths.test_root / "test.csv", paths.test_root)
    test_loader = build_test_dataloader(test_records, vocab=vocab)

    all_hyps = []
    model.eval()
    with torch.no_grad():
        for batch in test_loader:
            audio = batch.audio.to(device)
            audio_lengths = batch.audio_lengths.to(device)
            mel = mel_extractor(audio)
            mel_lengths = (
                (audio_lengths // FIXED["hop_length"] + 1).clamp(max=mel.size(-1)).long()
            )
            out = model(mel, mel_lengths)
            hyps = greedy_decode(out.log_probs, out.output_lengths, vocab)
            all_hyps.extend(hyps)

    # Pair filenames (preserved order from SequentialSampler) with predictions.
    pairs = [
        (rec.audio_path.name, words_to_digits(h))
        for rec, h in zip(test_records, all_hyps)
    ]
    submission_path = run_root / "submission.csv"
    write_submission(pairs, submission_path)
    print("Submission saved:", submission_path)
else:
    print("No test_root — skipping submission.")
